# Synthetic CMAB regime-matched fairness

This notebook runs and analyzes the synthetic contextual multi-armed bandit experiments

The objective is to evaluate whether fairness-aware bandit policies and reweighting-based preprocessing improve fairness metrics without causing an excessive loss in predictive utility.

Three synthetic regimes are considered:

- **Stationary deterministic regime**: evaluated with LinUCB and FairLinUCB;
- **Stationary stochastic regime**: evaluated with LinTS and FairLinTS;
- **Adversarial switching regime**: evaluated with EXP4 and FairEXP4.

For each regime, the notebook compares two preprocessing settings:

- **Uniform**: all observations receive equal update weight;
- **Reweighting**: observations are weighted according to their group.

The main utility metrics are average reward and cumulative prediction error. Fairness is assessed using Demographic Parity Gap, Equalized Odds Gap, and UtilityGap.

## Imports and portable project setup

This cell loads the project modules and detects the repository root automatically. The notebook is intended to work from a fresh GitHub clone without hard-coded local paths.

Most implementation details are the `src/fair_bandits/` package:

- synthetic data generation and preprocessing weights are handled in `fair_bandits.data`;
- policy construction and replay are handled through `fair_bandits.policies` and `fair_bandits.experiments`;
- metric normalization and temporal loading are handled in `fair_bandits.metrics` and `fair_bandits.io`;
- figure generation is handled in `fair_bandits.plots`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

def find_project_root(start: Path | None = None) -> Path:
    """
    Find the repository root from the current notebook location.

    This lets the notebook run both from VS Code and from Jupyter, whether the
    current working directory is the project root or a subfolder such as notebooks/.
    """
    current = Path.cwd().resolve() if start is None else Path(start).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src" / "fair_bandits").exists():
            return candidate
        if (candidate / "fair_bandits").exists():
            return candidate

    return current


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"

if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from fair_bandits.config import build_config

from fair_bandits.data import (
    make_imbalanced_synthetic_cmab_dataset,
)

from fair_bandits.metrics import normalize_metric_columns

from fair_bandits.policies import SyntheticPolicyParams

from fair_bandits.experiments import (
    run_imbalanced_synthetic_preflight,
    run_online_synthetic_benchmarks,
    run_synthetic_preflight,
)

from fair_bandits.io import (
    export_synthetic_final_summary_tables,
    export_synthetic_significance_tables,
    load_downsampled_synthetic_temporal_logs,
)

from fair_bandits.plots import (
    plot_synthetic_inprocessing_fairness,
    plot_synthetic_performance_metric,
    plot_synthetic_preprocessing_fairness,
)

from fair_bandits.plots import plot_synthetic_tradeoff_set

## Configuration

This section defines the experimental design and output locations.

The notebook can be run in two modes:

- **dev**: a short run used to check that the full pipeline works;
- **full**: the complete experiment used for thesis figures and tables.

The synthetic benchmark is organized by regime. Each regime is paired with the algorithm family that best matches its learning setting: LinUCB for deterministic environments, Thompson Sampling for stochastic environments, and EXP4 for adversarial expert-advice environments.

The output structure is also defined here. Raw trajectories and compact endpoint summaries are stored under the run directory, while final thesis-ready figures and tables are exported to separate `final_figures` and `tables` folders.

In [ ]:
# Use "dev" first. Switch to "full" only after the preflight and dev run succeed.
RUN_MODE = "full"  # "dev" or "full"

try:
    CFG = build_config(
        run_mode="full" if RUN_MODE == "full" else "dev",
    )
except TypeError:
    CFG = build_config(
        "full" if RUN_MODE == "full" else "dev",
    )

RESULTS_ROOT = Path(CFG.results_dir)
SYNTHETIC_ROOT = RESULTS_ROOT / "synthetic_cmab_regime"
RUN_DIR = SYNTHETIC_ROOT / RUN_MODE

TRAJECTORY_DIR = RUN_DIR / "trajectories"
RAW_TABLE_DIR = RUN_DIR / "tables"
METADATA_DIR = RUN_DIR / "metadata"

FIG_DIR = RESULTS_ROOT / "final_figures" / "synthetic"
TABLE_DIR = RESULTS_ROOT / "tables" / "synthetic"

for directory in [
    RUN_DIR,
    TRAJECTORY_DIR,
    RAW_TABLE_DIR,
    METADATA_DIR,
    FIG_DIR,
    TABLE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

REGIMES = [
    "stationary_deterministic",
    "stationary_stochastic",
    "adversarial_switching",
]

PREPROCESSINGS = [
    "uniform",
    "reweigh_group_label",
]

POLICIES_BY_REGIME = {
    "stationary_deterministic": [
        "LinUCB",
        "FairLinUCB_DP",
    ],
    "stationary_stochastic": [
        "LinTS",
        "FairLinTS_DP",
    ],
    "adversarial_switching": [
        "EXP4",
        "FairEXP4_DP",
    ],
}

REGIME_LABELS = {
    "stationary_deterministic": "Stationary deterministic",
    "stationary_stochastic": "Stationary stochastic",
    "adversarial_switching": "Adversarial switching",
}

REGIME_FILE_TAGS = {
    "stationary_deterministic": "deterministic",
    "stationary_stochastic": "stochastic",
    "adversarial_switching": "adversarial",
}

POLICY_LABELS = {
    "LinUCB": "LinUCB",
    "FairLinUCB_DP": "FairLinUCB",
    "LinTS": "LinTS",
    "FairLinTS_DP": "FairLinTS",
    "EXP4": "EXP4",
    "FairEXP4_DP": "FairEXP4",
}

PREPROCESSING_LABELS = {
    "uniform": "Uniform",
    "reweigh_group_label": "Reweighting",
}

if RUN_MODE == "dev":
    T_MAX = 1000
    CHECKPOINTS = [250, 500, 1000]
    SEEDS = [0, 1]
else:
    T_MAX = 5000
    CHECKPOINTS = [500, 1000, 2000, 5000]
    SEEDS = list(range(int(getattr(CFG, "n_seeds", 50))))

D = 10
N_EXPERTS = 6

# Policy hyperparameters
ALPHA_LINUCB = 1.5
LAMBDA_RIDGE = 1.0
TS_V = 0.5
EXP4_GAMMA = 0.07

DP_TAU = 0.02
DP_LAMBDA_LINEAR = 2.0
DP_LAMBDA_EXP4 = 0.20
BETA_SMOOTH = 1.0
MIN_GROUP_COUNT = 20

PLOT_EVERY = 10
ZOOM_START = 250

# Preserve existing trajectories unless an explicit rerun is requested.
FORCE_RERUN = False
RUN_PREFLIGHT = True
RUN_BENCHMARK = False

ENDPOINT_PATH = RAW_TABLE_DIR / "endpoint_perseed.csv"
RUN_INDEX_PATH = RAW_TABLE_DIR / "run_index.csv"

print("RUN_MODE:", RUN_MODE)
print("RUN_DIR:", RUN_DIR)
print("T_MAX:", T_MAX)
print("CHECKPOINTS:", CHECKPOINTS)
print("Number of seeds:", len(SEEDS))
print("FORCE_RERUN:", FORCE_RERUN)
print("RUN_BENCHMARK:", RUN_BENCHMARK)

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# LOAD ORIGINAL FULL SYNTHETIC RESULTS
# ============================================================

OLD_SYNTHETIC_TABLE_DIR = Path(
    r"C:\Users\jmgre\Documents\Python_Tests\Fairness-JM-JN\notebooks"
    r"\results\full\synthetic_cmab_regime\full\tables"
)

ENDPOINT_PATH = (
    OLD_SYNTHETIC_TABLE_DIR
    / "endpoint_perseed.csv"
)

RUN_INDEX_PATH = (
    OLD_SYNTHETIC_TABLE_DIR
    / "run_index.csv"
)

print("ENDPOINT_PATH:")
print(ENDPOINT_PATH)
print("Exists:", ENDPOINT_PATH.exists())

print("\nRUN_INDEX_PATH:")
print(RUN_INDEX_PATH)
print("Exists:", RUN_INDEX_PATH.exists())

assert ENDPOINT_PATH.exists(), "endpoint_perseed.csv not found"
assert RUN_INDEX_PATH.exists(), "run_index.csv not found"


# Load existing results
endpoint_df = normalize_metric_columns(
    pd.read_csv(ENDPOINT_PATH)
)

run_index_df = pd.read_csv(
    RUN_INDEX_PATH
)

print("\nLoaded successfully")
print("endpoint_df shape:", endpoint_df.shape)
print("run_index_df shape:", run_index_df.shape)

print("\nRegimes:")
print(sorted(endpoint_df["regime"].unique()))

print("\nPolicies:")
print(sorted(endpoint_df["policy"].unique()))

print("\nPreprocessings:")
print(sorted(endpoint_df["preprocessing"].unique()))

print("\nSeeds:", endpoint_df["seed"].nunique())
print("Checkpoints:", sorted(endpoint_df["T"].unique()))

## Preflight

The preflight is a small sanity check before running or loading the full benchmark. For each synthetic regime, it generates a short environment, computes the Reweighting support table, and replays one fairness-aware policy.

This step checks that:

- the synthetic generator works for all regimes;
- group and oracle-action strata are correctly represented;
- Reweighting weights are well-defined;
- EXP4 expert advice is correctly generated for the adversarial regime;
- the replay function produces the expected temporal metrics.

In [ ]:
preflight_params = SyntheticPolicyParams(
    d=D,
    alpha_linucb=ALPHA_LINUCB,
    lambda_ridge=LAMBDA_RIDGE,
    ts_v=TS_V,
    exp4_gamma=EXP4_GAMMA,
    dp_tau=DP_TAU,
    dp_lambda_linear=DP_LAMBDA_LINEAR,
    dp_lambda_exp4=DP_LAMBDA_EXP4,
    beta_smooth=BETA_SMOOTH,
    min_group_count=MIN_GROUP_COUNT,
    n_experts=N_EXPERTS,
)

if RUN_PREFLIGHT:
    preflight_results = run_synthetic_preflight(
        regimes=REGIMES,
        policies_by_regime=POLICIES_BY_REGIME,
        params=preflight_params,
        d=D,
        n_experts=N_EXPERTS,
        t=250,
        seed=42,
        preprocessing="reweigh_group_label",
        display_fn=display,
    )
else:
    print("Preflight skipped because RUN_PREFLIGHT is False.")

## Launch or reload benchmark outputs

This cell either runs the synthetic benchmark or reloads existing outputs.

When `RUN_BENCHMARK = True`, the benchmark evaluates all combinations of:

- regime;
- preprocessing method;
- regime-matched policy;
- random seed;
- evaluation checkpoint.

For each trajectory, the benchmark stores a compressed temporal log. It also exports two compact files:

- `endpoint_perseed.csv`: one row per seed and checkpoint, used for tables and statistical tests;
- `run_index.csv`: an index of generated trajectories, used to reload temporal curves efficiently.

When `RUN_BENCHMARK = False`, the notebook reloads these files instead of recomputing the benchmark.

In [ ]:
params = SyntheticPolicyParams(
    d=D,
    alpha_linucb=ALPHA_LINUCB,
    lambda_ridge=LAMBDA_RIDGE,
    ts_v=TS_V,
    exp4_gamma=EXP4_GAMMA,
    dp_tau=DP_TAU,
    dp_lambda_linear=DP_LAMBDA_LINEAR,
    dp_lambda_exp4=DP_LAMBDA_EXP4,
    beta_smooth=BETA_SMOOTH,
    min_group_count=MIN_GROUP_COUNT,
    n_experts=N_EXPERTS,
)

if RUN_BENCHMARK:
    endpoint_df, run_index_df = run_online_synthetic_benchmarks(
        run_dir=RUN_DIR,
        regimes=REGIMES,
        preprocessings=PREPROCESSINGS,
        policies_by_regime=POLICIES_BY_REGIME,
        seeds=SEEDS,
        checkpoints=CHECKPOINTS,
        t_max=T_MAX,
        params=params,
        force_rerun=FORCE_RERUN,
    )

    endpoint_df = normalize_metric_columns(endpoint_df)

    print("\nEndpoint rows:", len(endpoint_df))
    print("Run-index rows:", len(run_index_df))

    print("\nEndpoint preview:")
    print(endpoint_df.head().to_string(index=False))

    print("\nRun-index preview:")
    print(run_index_df.head().to_string(index=False))

else:
    print(
        "Benchmark generation is disabled. "
        "Set RUN_BENCHMARK = True in the configuration cell when ready."
    )

    if ENDPOINT_PATH.exists() and RUN_INDEX_PATH.exists():
        endpoint_df = normalize_metric_columns(pd.read_csv(ENDPOINT_PATH))
        run_index_df = pd.read_csv(RUN_INDEX_PATH)

        print("\nLoaded existing outputs:")
        print("endpoint_df:", endpoint_df.shape)
        print("run_index_df:", run_index_df.shape)

        print("\nEndpoint preview:")
        print(endpoint_df.head().to_string(index=False))

        print("\nRun-index preview:")
        print(run_index_df.head().to_string(index=False))

    else:
        print("\nNo existing benchmark outputs found yet.")
        print("ENDPOINT_PATH:", ENDPOINT_PATH)
        print("RUN_INDEX_PATH:", RUN_INDEX_PATH)

## Validate compact output files

This section checks that the benchmark outputs have the expected structure before any figures or tables are produced.

The validation verifies:

- the number of rows in the endpoint and run-index files;
- the list of regimes, policies, preprocessings, seeds, and checkpoints;
- the presence of key utility and fairness columns;
- compatibility with the thesis terminology, in particular `cumulative_prediction_error`.


In [ ]:
if "endpoint_df" not in globals() or "run_index_df" not in globals():
    raise RuntimeError(
        "No endpoint/run-index data available. "
        "Run the benchmark first, or point RUN_DIR to existing outputs."
    )

endpoint_df = normalize_metric_columns(endpoint_df)

print("endpoint_df shape:", endpoint_df.shape)
print("run_index_df shape:", run_index_df.shape)
print("\nRegimes:", sorted(endpoint_df["regime"].unique()))
print("Preprocessings:", sorted(endpoint_df["preprocessing"].unique()))
print("Policies:", sorted(endpoint_df["policy"].unique()))
print("Seeds:", endpoint_df["seed"].nunique())
print("Checkpoints:", sorted(endpoint_df["T"].unique()))

expected_endpoint_rows = (
    len(REGIMES)
    * len(PREPROCESSINGS)
    * 2
    * len(SEEDS)
    * len(CHECKPOINTS)
)

expected_run_rows = (
    len(REGIMES)
    * len(PREPROCESSINGS)
    * 2
    * len(SEEDS)
)

print("\nExpected endpoint rows:", expected_endpoint_rows)
print("Observed endpoint rows:", len(endpoint_df))
print("Expected run-index rows:", expected_run_rows)
print("Observed run-index rows:", len(run_index_df))

required_endpoint_cols = [
    "regime",
    "preprocessing",
    "policy",
    "seed",
    "T",
    "accuracy",
    "avg_reward",
    "cumulative_prediction_error",
    "DP_gap",
    "EO_gap",
    "UtilityGap",
]

missing_endpoint_cols = [
    column
    for column in required_endpoint_cols
    if column not in endpoint_df.columns
]

if missing_endpoint_cols:
    raise ValueError(f"Missing endpoint columns: {missing_endpoint_cols}")

if len(endpoint_df) != expected_endpoint_rows:
    print(
        "Warning: endpoint row count differs from the expected run. "
        "Inspect the file before using the tables in the thesis."
    )

if len(run_index_df) != expected_run_rows:
    print(
        "Warning: run-index row count differs from the expected run. "
        "Inspect the file before using the figures in the thesis."
    )

display(endpoint_df.head())
display(run_index_df.head())

In [ ]:
# ============================================================
# FINAL ENDPOINT AT T = 5000
# ============================================================

FINAL_T = int(endpoint_df["T"].max())

final_endpoint_df = (
    endpoint_df[
        endpoint_df["T"] == FINAL_T
    ]
    .copy()
)

final_endpoint_df = normalize_metric_columns(
    final_endpoint_df
)

print("FINAL_T:", FINAL_T)
print("Final endpoint rows:", len(final_endpoint_df))
print("Seeds:", final_endpoint_df["seed"].nunique())

print("\nRows by regime / policy / preprocessing:")
display(
    final_endpoint_df
    .groupby(
        [
            "regime",
            "policy",
            "preprocessing",
        ]
    )["seed"]
    .nunique()
    .to_frame("n_seeds")
)

## Load downsampled temporal trajectories

This section reloads the compressed trajectory files listed in the run index and keeps only regularly spaced time points for plotting.

The full online trajectories can be large, especially in the full run with many seeds. Downsampling reduces memory usage and figure-generation time while preserving the overall temporal patterns of the learning curves.

The resulting dataframe is used only for temporal figures. Final summary tables and statistical tests are based on the compact endpoint table.


In [ ]:
temporal_df = load_downsampled_synthetic_temporal_logs(
    run_index_df,
    run_dir=RUN_DIR,
    plot_every=PLOT_EVERY,
)

print("\ntemporal_df shape:", temporal_df.shape)

print(
    "Rows per trajectory:",
    temporal_df.groupby(
        ["regime", "preprocessing", "policy", "seed"]
    ).size().unique(),
)

print(temporal_df.head().to_string(index=False))

## Figures

For each regime, five figures are produced:

1. **Preprocessing fairness comparison**: DP and EO gaps for uniform versus Reweighting.
2. **In-processing fairness comparison**: baseline versus fairness-aware policy under uniform preprocessing.
3. **Average reward over time**.
4. **Cumulative prediction error over time**.
5. **UtilityGap over time**.

The figures focus on the learning dynamics after the initial warm-up period, to make interpretation easier and avoid over-emphasizing early instability in the first rounds.

Temporal bands indicate pointwise 95% confidence intervals for the mean trajectory across seeds.

In [ ]:
figure_paths = []

for regime in REGIMES:
    print("Generating figures for:", regime)

    policies = POLICIES_BY_REGIME[regime]

    figure_paths.append(
        plot_synthetic_preprocessing_fairness(
            temporal_df,
            regime=regime,
            output_dir=FIG_DIR,
            policies=policies,
            preprocessings=PREPROCESSINGS,
            regime_label=REGIME_LABELS[regime],
            regime_file_tag=REGIME_FILE_TAGS[regime],
            policy_labels=POLICY_LABELS,
            preprocessing_labels=PREPROCESSING_LABELS,
            zoom_start=ZOOM_START,
            show=True,
        )
    )

    figure_paths.append(
        plot_synthetic_inprocessing_fairness(
            temporal_df,
            regime=regime,
            output_dir=FIG_DIR,
            policies=policies,
            regime_label=REGIME_LABELS[regime],
            regime_file_tag=REGIME_FILE_TAGS[regime],
            policy_labels=POLICY_LABELS,
            zoom_start=ZOOM_START,
            show=True,
        )
    )

    figure_paths.append(
        plot_synthetic_performance_metric(
            temporal_df,
            regime=regime,
            metric_col="avg_reward",
            ylabel="Average reward",
            filename_suffix="average_reward",
            output_dir=FIG_DIR,
            policies=policies,
            preprocessings=PREPROCESSINGS,
            regime_label=REGIME_LABELS[regime],
            regime_file_tag=REGIME_FILE_TAGS[regime],
            policy_labels=POLICY_LABELS,
            preprocessing_labels=PREPROCESSING_LABELS,
            zoom_start=ZOOM_START,
            show=True,
        )
    )

    figure_paths.append(
        plot_synthetic_performance_metric(
            temporal_df,
            regime=regime,
            metric_col="cumulative_prediction_error",
            ylabel="Cumulative prediction error",
            filename_suffix="cumulative_prediction_error",
            output_dir=FIG_DIR,
            policies=policies,
            preprocessings=PREPROCESSINGS,
            regime_label=REGIME_LABELS[regime],
            regime_file_tag=REGIME_FILE_TAGS[regime],
            policy_labels=POLICY_LABELS,
            preprocessing_labels=PREPROCESSING_LABELS,
            zoom_start=ZOOM_START,
            show=True,
        )
    )

    figure_paths.append(
        plot_synthetic_performance_metric(
            temporal_df,
            regime=regime,
            metric_col="UtilityGap_over_time",
            ylabel="UtilityGap",
            filename_suffix="utility_gap",
            output_dir=FIG_DIR,
            policies=policies,
            preprocessings=PREPROCESSINGS,
            regime_label=REGIME_LABELS[regime],
            regime_file_tag=REGIME_FILE_TAGS[regime],
            policy_labels=POLICY_LABELS,
            preprocessing_labels=PREPROCESSING_LABELS,
            zoom_start=ZOOM_START,
            show=True,
        )
    )

print("\nGenerated figures:", len(figure_paths))

for path in figure_paths:
    print(path)

## Trade-off plots

In [ ]:
SYNTHETIC_TRADEOFF_DIR = FIG_DIR / "tradeoff"

synthetic_tradeoff_paths = plot_synthetic_tradeoff_set(
    temporal_df=temporal_df,
    regimes=REGIMES,
    policies_by_regime=POLICIES_BY_REGIME,
    regime_labels=REGIME_LABELS,
    regime_file_tags=REGIME_FILE_TAGS,
    preprocessings=PREPROCESSINGS,
    fig_dir=SYNTHETIC_TRADEOFF_DIR,
    policy_labels=POLICY_LABELS,
    preprocessing_labels=PREPROCESSING_LABELS,
    confidence=0.95,
    show=True,
)

print("Synthetic trade-off figures:", len(synthetic_tradeoff_paths))
for path in synthetic_tradeoff_paths:
    print(path)

## Final endpoint data

This cell extracts the final checkpoint from the compact endpoint table.

The final checkpoint corresponds to the last evaluation horizon of the selected run mode. It is used for summary tables and paired statistical tests.

At this stage, each row represents one seed for one combination of regime, preprocessing, and policy.

In [ ]:
FINAL_T = int(endpoint_df["T"].max())

final_endpoint_df = endpoint_df[
    endpoint_df["T"] == FINAL_T
].copy()

final_endpoint_df = normalize_metric_columns(
    final_endpoint_df
)

print("FINAL_T:", FINAL_T)
print("Final endpoint rows:", len(final_endpoint_df))
print("Seeds:", final_endpoint_df["seed"].nunique())

display(final_endpoint_df.head())

## Final utility and fairness summary tables

This section creates compact final summary tables for each synthetic regime.

The tables report:

- **Average reward**: higher values indicate better predictive utility;
- **Cumulative prediction error**: lower values indicate fewer accumulated prediction errors relative to the oracle action probabilities;
- **UtilityGap**: lower values indicate smaller between-group disparities in achieved reward.

Values are reported as mean ± standard deviation across seeds at the final checkpoint. This table convention is distinct from the temporal figures, where shaded bands represent 95% confidence intervals for the mean trajectory. These tables are designed to summarize the utility–fairness trade-off without overloading the main text with all temporal curves.

In [ ]:
SUMMARY_METRICS = [
    ("avg_reward", "Average reward"),
    ("cumulative_prediction_error", "Cumulative prediction error"),
    ("UtilityGap", "UtilityGap"),
]

summary_tables = export_synthetic_final_summary_tables(
    final_endpoint_df=final_endpoint_df,
    regimes=REGIMES,
    table_dir=TABLE_DIR,
    policies_by_regime=POLICIES_BY_REGIME,
    preprocessings=PREPROCESSINGS,
    summary_metrics=SUMMARY_METRICS,
    regime_labels=REGIME_LABELS,
    regime_file_tags=REGIME_FILE_TAGS,
    final_t=FINAL_T,
    policy_labels=POLICY_LABELS,
    preprocessing_labels=PREPROCESSING_LABELS,
    caption_prefix="Final utility and fairness summary for the synthetic",
    label_prefix="tab:synthetic",
    file_prefix="synthetic",
    digits=4,
)

## Paired Wilcoxon tests with Holm correction

This section performs paired statistical comparisons across seeds at the final checkpoint.

The comparisons are paired by seed, so each intervention is compared against its corresponding baseline on the same synthetic environment. This reduces variability due to random environment generation.

The tested metrics are DP gap, EO gap, UtilityGap, average reward, and cumulative prediction error. Holm correction is applied within each regime to control for multiple comparisons.

The resulting tables are intended as supporting evidence for the main descriptive figures and summary tables.

In [ ]:
SIGNIFICANCE_METRICS = [
    ("DP_gap", "DP gap"),
    ("EO_gap", "EO gap"),
    ("UtilityGap", "UtilityGap"),
    ("average_reward", "Average reward"),
    ("cumulative_prediction_error", "Cumulative prediction error"),
]

SIGNIFICANCE_COMPARISONS_BY_REGIME = {
    "stationary_deterministic": [
        {
            "Comparison": "Preprocessing: LinUCB uniform vs reweighting",
            "baseline_policy": "LinUCB",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "LinUCB",
            "intervention_preprocessing": "reweigh_group_label",
        },
        {
            "Comparison": "In-processing: LinUCB vs FairLinUCB",
            "baseline_policy": "LinUCB",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "FairLinUCB_DP",
            "intervention_preprocessing": "uniform",
        },
    ],
    "stationary_stochastic": [
        {
            "Comparison": "Preprocessing: LinTS uniform vs reweighting",
            "baseline_policy": "LinTS",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "LinTS",
            "intervention_preprocessing": "reweigh_group_label",
        },
        {
            "Comparison": "In-processing: LinTS vs FairLinTS",
            "baseline_policy": "LinTS",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "FairLinTS_DP",
            "intervention_preprocessing": "uniform",
        },
    ],
    "adversarial_switching": [
        {
            "Comparison": "Preprocessing: EXP4 uniform vs reweighting",
            "baseline_policy": "EXP4",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "EXP4",
            "intervention_preprocessing": "reweigh_group_label",
        },
        {
            "Comparison": "In-processing: EXP4 vs FairEXP4",
            "baseline_policy": "EXP4",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "FairEXP4_DP",
            "intervention_preprocessing": "uniform",
        },
    ],
}

significance_tables = export_synthetic_significance_tables(
    final_endpoint_df=final_endpoint_df,
    regimes=REGIMES,
    table_dir=TABLE_DIR,
    comparisons_by_regime=SIGNIFICANCE_COMPARISONS_BY_REGIME,
    significance_metrics=SIGNIFICANCE_METRICS,
    regime_labels=REGIME_LABELS,
    regime_file_tags=REGIME_FILE_TAGS,
    final_t=FINAL_T,
    policy_labels=POLICY_LABELS,
    preprocessing_labels=PREPROCESSING_LABELS,
    caption_prefix="Paired Wilcoxon tests with Holm correction for the synthetic",
    label_prefix="tab:synthetic",
    file_prefix="synthetic",
    seed_col="seed",
    alpha=0.05,
    value_digits=3,
)

## Sensitivity analysis — regime-matched synthetic experiments with 80/20 group imbalance

This section repeats the synthetic CMAB experiments under an imbalanced group distribution. The majority group represents 80% of the observations and the minority group represents 20%.

The objective is to check whether the conclusions observed in the balanced synthetic regimes remain stable when the sensitive groups are unequally represented.

The analysis remains regime-matched:
- stationary deterministic regime: LinUCB and FairLinUCB;
- stationary stochastic regime: LinTS and FairLinTS;
- adversarial switching regime: EXP4 and FairEXP4.

The same two preprocessing settings are used: uniform weighting and group–oracle-action reweighting.

In [ ]:
RUN_IMBALANCED_PREFLIGHT = True
RUN_IMBALANCED_BENCHMARK = True
IMBALANCED_FORCE_RERUN = False

IMBALANCED_MINORITY_FRACTION = 0.20

IMBALANCED_SCENARIO_TAG = "synthetic_imbalanced_80_20_regime_matched"

IMBALANCED_REGIMES = [
    "stationary_deterministic",
    "stationary_stochastic",
    "adversarial_switching",
]

IMBALANCED_POLICIES_BY_REGIME = {
    "stationary_deterministic": [
        "LinUCB",
        "FairLinUCB_DP",
    ],
    "stationary_stochastic": [
        "LinTS",
        "FairLinTS_DP",
    ],
    "adversarial_switching": [
        "EXP4",
        "FairEXP4_DP",
    ],
}

IMBALANCED_REGIME_LABELS = {
    "stationary_deterministic": "Stationary deterministic — imbalanced groups (80/20)",
    "stationary_stochastic": "Stationary stochastic — imbalanced groups (80/20)",
    "adversarial_switching": "Adversarial switching — imbalanced groups (80/20)",
}

IMBALANCED_REGIME_FILE_TAGS = {
    "stationary_deterministic": "deterministic_imbalanced_80_20",
    "stationary_stochastic": "stochastic_imbalanced_80_20",
    "adversarial_switching": "adversarial_imbalanced_80_20",
}

IMBALANCED_SYNTHETIC_ROOT = (
    Path(CFG.results_dir)
    / "synthetic_cmab_sensitivity_80_20_regime"
)

IMBALANCED_RUN_DIR = IMBALANCED_SYNTHETIC_ROOT / RUN_MODE

IMBALANCED_FIG_DIR = (
    RESULTS_ROOT
    / "final_figures"
    / "synthetic"
    / "sensitivity_80_20_regime_matched"
)

IMBALANCED_TABLE_DIR = (
    RESULTS_ROOT
    / "tables"
    / "synthetic"
    / "sensitivity_80_20_regime_matched"
)

IMBALANCED_ENDPOINT_PATH = (
    IMBALANCED_RUN_DIR
    / "tables"
    / "endpoint_perseed.csv"
)

IMBALANCED_RUN_INDEX_PATH = (
    IMBALANCED_RUN_DIR
    / "tables"
    / "run_index.csv"
)

for directory in [
    IMBALANCED_RUN_DIR,
    IMBALANCED_FIG_DIR,
    IMBALANCED_TABLE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("IMBALANCED_RUN_DIR:", IMBALANCED_RUN_DIR)
print("IMBALANCED_FIG_DIR:", IMBALANCED_FIG_DIR)
print("IMBALANCED_TABLE_DIR:", IMBALANCED_TABLE_DIR)
print("RUN_IMBALANCED_BENCHMARK:", RUN_IMBALANCED_BENCHMARK)
print("IMBALANCED_MINORITY_FRACTION:", IMBALANCED_MINORITY_FRACTION)

In [ ]:
imbalanced_preflight_params = SyntheticPolicyParams(
    d=D,
    alpha_linucb=ALPHA_LINUCB,
    lambda_ridge=LAMBDA_RIDGE,
    ts_v=TS_V,
    exp4_gamma=EXP4_GAMMA,
    dp_tau=DP_TAU,
    dp_lambda_linear=DP_LAMBDA_LINEAR,
    dp_lambda_exp4=DP_LAMBDA_EXP4,
    beta_smooth=BETA_SMOOTH,
    min_group_count=MIN_GROUP_COUNT,
    n_experts=N_EXPERTS,
)

if RUN_IMBALANCED_PREFLIGHT:
    imbalanced_preflight_results = run_imbalanced_synthetic_preflight(
        regimes=IMBALANCED_REGIMES,
        policies_by_regime=IMBALANCED_POLICIES_BY_REGIME,
        params=imbalanced_preflight_params,
        d=D,
        n_experts=N_EXPERTS,
        minority_fraction=IMBALANCED_MINORITY_FRACTION,
        t=250,
        seed=42,
        preprocessing="reweigh_group_label",
        display_fn=display,
    )
else:
    print("80/20 preflight skipped.")

## Run or load the 80/20 sensitivity benchmark

This section either runs the regime-matched 80/20 sensitivity benchmark or reloads existing cached outputs.

The 80/20 benchmark follows the same structure as the balanced synthetic benchmark, but the generated environment enforces an imbalanced sensitive-group distribution. The majority group represents 80% of the observations and the minority group represents 20%.

The goal is to evaluate whether the behaviour observed in the balanced synthetic setting remains stable when one sensitive group is under-represented.

In [ ]:
def make_current_imbalanced_synthetic_dataset(**kwargs):
    return make_imbalanced_synthetic_cmab_dataset(
        **kwargs,
        minority_fraction=IMBALANCED_MINORITY_FRACTION,
    )

imbalanced_params = SyntheticPolicyParams(
    d=D,
    alpha_linucb=ALPHA_LINUCB,
    lambda_ridge=LAMBDA_RIDGE,
    ts_v=TS_V,
    exp4_gamma=EXP4_GAMMA,
    dp_tau=DP_TAU,
    dp_lambda_linear=DP_LAMBDA_LINEAR,
    dp_lambda_exp4=DP_LAMBDA_EXP4,
    beta_smooth=BETA_SMOOTH,
    min_group_count=MIN_GROUP_COUNT,
    n_experts=N_EXPERTS,
)

if RUN_IMBALANCED_BENCHMARK:
    imbalanced_endpoint_df, imbalanced_run_index_df = run_online_synthetic_benchmarks(
        run_dir=IMBALANCED_RUN_DIR,
        regimes=IMBALANCED_REGIMES,
        preprocessings=PREPROCESSINGS,
        policies_by_regime=IMBALANCED_POLICIES_BY_REGIME,
        seeds=SEEDS,
        checkpoints=CHECKPOINTS,
        t_max=T_MAX,
        params=imbalanced_params,
        force_rerun=IMBALANCED_FORCE_RERUN,
        dataset_factory=make_current_imbalanced_synthetic_dataset,
    )

    imbalanced_endpoint_df = normalize_metric_columns(imbalanced_endpoint_df)

else:
    if IMBALANCED_ENDPOINT_PATH.exists() and IMBALANCED_RUN_INDEX_PATH.exists():
        imbalanced_endpoint_df = normalize_metric_columns(
            pd.read_csv(IMBALANCED_ENDPOINT_PATH)
        )
        imbalanced_run_index_df = pd.read_csv(IMBALANCED_RUN_INDEX_PATH)

        print("Loaded existing 80/20 outputs:")
        print("imbalanced_endpoint_df:", imbalanced_endpoint_df.shape)
        print("imbalanced_run_index_df:", imbalanced_run_index_df.shape)

    else:
        raise FileNotFoundError(
            "No 80/20 sensitivity outputs found. "
            "Set RUN_IMBALANCED_BENCHMARK = True to generate them."
        )

print("\n80/20 endpoint preview:")
print(imbalanced_endpoint_df.head().to_string(index=False))

print("\n80/20 run-index preview:")
print(imbalanced_run_index_df.head().to_string(index=False))

## Validate 80/20 benchmark outputs

This section checks that the 80/20 sensitivity outputs have the expected structure before figures and tables are generated.

The validation verifies the number of endpoint rows, the number of run-index rows, the available regimes, policies, preprocessing settings, seeds, and final checkpoints. This helps detect incomplete cached runs before producing summary figures or statistical comparisons.

In [ ]:
imbalanced_endpoint_df = normalize_metric_columns(imbalanced_endpoint_df)

print("imbalanced_endpoint_df shape:", imbalanced_endpoint_df.shape)
print("imbalanced_run_index_df shape:", imbalanced_run_index_df.shape)

print("\nRegimes:", sorted(imbalanced_endpoint_df["regime"].unique()))
print("Preprocessings:", sorted(imbalanced_endpoint_df["preprocessing"].unique()))
print("Policies:", sorted(imbalanced_endpoint_df["policy"].unique()))
print("Seeds:", imbalanced_endpoint_df["seed"].nunique())
print("Checkpoints:", sorted(imbalanced_endpoint_df["T"].unique()))

n_imbalanced_policy_settings = sum(
    len(IMBALANCED_POLICIES_BY_REGIME[regime])
    for regime in IMBALANCED_REGIMES
)

expected_imbalanced_endpoint_rows = (
    n_imbalanced_policy_settings
    * len(PREPROCESSINGS)
    * len(SEEDS)
    * len(CHECKPOINTS)
)

expected_imbalanced_run_rows = (
    n_imbalanced_policy_settings
    * len(PREPROCESSINGS)
    * len(SEEDS)
)

print("\nExpected endpoint rows:", expected_imbalanced_endpoint_rows)
print("Observed endpoint rows:", len(imbalanced_endpoint_df))
print("Expected run-index rows:", expected_imbalanced_run_rows)
print("Observed run-index rows:", len(imbalanced_run_index_df))

if len(imbalanced_endpoint_df) != expected_imbalanced_endpoint_rows:
    print("Warning: unexpected number of 80/20 endpoint rows.")

if len(imbalanced_run_index_df) != expected_imbalanced_run_rows:
    print("Warning: unexpected number of 80/20 run-index rows.")

display(imbalanced_endpoint_df.head())
display(imbalanced_run_index_df.head())

## Load 80/20 temporal trajectories

This section reloads the temporal trajectories generated for the 80/20 sensitivity analysis.

As in the balanced benchmark, the trajectories are downsampled for plotting. The temporal dataframe is used to visualize how utility and fairness metrics evolve over time under group imbalance.

In [ ]:
imbalanced_temporal_df = load_downsampled_synthetic_temporal_logs(
    imbalanced_run_index_df,
    run_dir=IMBALANCED_RUN_DIR,
    plot_every=PLOT_EVERY,
)

print("\nimbalanced_temporal_df shape:", imbalanced_temporal_df.shape)

print(
    "Rows per trajectory:",
    imbalanced_temporal_df.groupby(
        ["regime", "preprocessing", "policy", "seed"]
    ).size().unique(),
)

print(imbalanced_temporal_df.head().to_string(index=False))

## Figures 80/20

This section generates the same five standard temporal figures for each imbalanced synthetic regime:

1. preprocessing comparison for Demographic Parity Gap and Equalized Odds Gap;
2. in-processing comparison under uniform preprocessing;
3. average reward over time;
4. cumulative prediction error over time;
5. UtilityGap over time.

Using the same figure structure as the balanced benchmark makes it easier to compare the effect of group imbalance across regimes.

Temporal bands indicate pointwise 95% confidence intervals for the mean trajectory across seeds.

In [ ]:
imbalanced_figure_paths = []

for regime in IMBALANCED_REGIMES:
    print("Generating 80/20 figures for:", regime)

    policies = IMBALANCED_POLICIES_BY_REGIME[regime]

    imbalanced_figure_paths.append(
        plot_synthetic_preprocessing_fairness(
            imbalanced_temporal_df,
            regime=regime,
            output_dir=IMBALANCED_FIG_DIR,
            policies=policies,
            preprocessings=PREPROCESSINGS,
            regime_label=IMBALANCED_REGIME_LABELS[regime],
            regime_file_tag=IMBALANCED_REGIME_FILE_TAGS[regime],
            policy_labels=POLICY_LABELS,
            preprocessing_labels=PREPROCESSING_LABELS,
            zoom_start=ZOOM_START,
            show=True,
        )
    )

    imbalanced_figure_paths.append(
        plot_synthetic_inprocessing_fairness(
            imbalanced_temporal_df,
            regime=regime,
            output_dir=IMBALANCED_FIG_DIR,
            policies=policies,
            regime_label=IMBALANCED_REGIME_LABELS[regime],
            regime_file_tag=IMBALANCED_REGIME_FILE_TAGS[regime],
            policy_labels=POLICY_LABELS,
            zoom_start=ZOOM_START,
            show=True,
        )
    )

    imbalanced_figure_paths.append(
        plot_synthetic_performance_metric(
            imbalanced_temporal_df,
            regime=regime,
            metric_col="avg_reward",
            ylabel="Average reward",
            filename_suffix="average_reward",
            output_dir=IMBALANCED_FIG_DIR,
            policies=policies,
            preprocessings=PREPROCESSINGS,
            regime_label=IMBALANCED_REGIME_LABELS[regime],
            regime_file_tag=IMBALANCED_REGIME_FILE_TAGS[regime],
            policy_labels=POLICY_LABELS,
            preprocessing_labels=PREPROCESSING_LABELS,
            zoom_start=ZOOM_START,
            show=True,
        )
    )

    imbalanced_figure_paths.append(
        plot_synthetic_performance_metric(
            imbalanced_temporal_df,
            regime=regime,
            metric_col="cumulative_prediction_error",
            ylabel="Cumulative prediction error",
            filename_suffix="cumulative_prediction_error",
            output_dir=IMBALANCED_FIG_DIR,
            policies=policies,
            preprocessings=PREPROCESSINGS,
            regime_label=IMBALANCED_REGIME_LABELS[regime],
            regime_file_tag=IMBALANCED_REGIME_FILE_TAGS[regime],
            policy_labels=POLICY_LABELS,
            preprocessing_labels=PREPROCESSING_LABELS,
            zoom_start=ZOOM_START,
            show=True,
        )
    )

    imbalanced_figure_paths.append(
        plot_synthetic_performance_metric(
            imbalanced_temporal_df,
            regime=regime,
            metric_col="UtilityGap_over_time",
            ylabel="UtilityGap",
            filename_suffix="utility_gap",
            output_dir=IMBALANCED_FIG_DIR,
            policies=policies,
            preprocessings=PREPROCESSINGS,
            regime_label=IMBALANCED_REGIME_LABELS[regime],
            regime_file_tag=IMBALANCED_REGIME_FILE_TAGS[regime],
            policy_labels=POLICY_LABELS,
            preprocessing_labels=PREPROCESSING_LABELS,
            zoom_start=ZOOM_START,
            show=True,
        )
    )

print("\nGenerated 80/20 figures:", len(imbalanced_figure_paths))

for path in imbalanced_figure_paths:
    print(path)

## 80/20 final summary tables

This section exports compact final summary tables for the 80/20 sensitivity analysis.

The tables report average reward, cumulative prediction error, and UtilityGap at the final checkpoint. They are designed to show whether the main utility and group-utility conclusions remain stable when the sensitive-group distribution is imbalanced.

As in the balanced synthetic benchmark, values are reported as mean ± standard deviation across seeds, while temporal figures use 95% confidence intervals for the mean trajectory.

In [ ]:
IMBALANCED_FINAL_T = int(imbalanced_endpoint_df["T"].max())

imbalanced_final_endpoint_df = imbalanced_endpoint_df[
    imbalanced_endpoint_df["T"] == IMBALANCED_FINAL_T
].copy()

imbalanced_final_endpoint_df = normalize_metric_columns(
    imbalanced_final_endpoint_df
)

print("IMBALANCED_FINAL_T:", IMBALANCED_FINAL_T)
print("Final 80/20 endpoint rows:", len(imbalanced_final_endpoint_df))
print("Seeds:", imbalanced_final_endpoint_df["seed"].nunique())

display(imbalanced_final_endpoint_df.head())

imbalanced_summary_tables = export_synthetic_final_summary_tables(
    final_endpoint_df=imbalanced_final_endpoint_df,
    regimes=IMBALANCED_REGIMES,
    table_dir=IMBALANCED_TABLE_DIR,
    policies_by_regime=IMBALANCED_POLICIES_BY_REGIME,
    preprocessings=PREPROCESSINGS,
    summary_metrics=SUMMARY_METRICS,
    regime_labels=IMBALANCED_REGIME_LABELS,
    regime_file_tags=IMBALANCED_REGIME_FILE_TAGS,
    final_t=IMBALANCED_FINAL_T,
    policy_labels=POLICY_LABELS,
    preprocessing_labels=PREPROCESSING_LABELS,
    caption_prefix=(
        "Final utility and fairness summary for the synthetic "
        "regime-matched 80/20 sensitivity analysis"
    ),
    label_prefix="tab:synthetic",
    file_prefix="synthetic",
    digits=4,
)

## 80/20 regime-matched sensitivity analysis — paired significance tests

This section performs paired Wilcoxon tests for the imbalanced 80/20 regime-matched sensitivity analysis.

The comparisons are paired by seed. The objective is to test whether reweighting or the fairness-aware in-processing policy significantly changes utility or fairness metrics at the final horizon.

Holm correction is applied across all tests in this 80/20 sensitivity-analysis table.

In [ ]:
IMBALANCED_SIGNIFICANCE_COMPARISONS_BY_REGIME = {
    "stationary_deterministic": [
        {
            "Comparison": "Preprocessing: LinUCB uniform vs reweighting",
            "baseline_policy": "LinUCB",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "LinUCB",
            "intervention_preprocessing": "reweigh_group_label",
        },
        {
            "Comparison": "In-processing: LinUCB vs FairLinUCB",
            "baseline_policy": "LinUCB",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "FairLinUCB_DP",
            "intervention_preprocessing": "uniform",
        },
    ],
    "stationary_stochastic": [
        {
            "Comparison": "Preprocessing: LinTS uniform vs reweighting",
            "baseline_policy": "LinTS",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "LinTS",
            "intervention_preprocessing": "reweigh_group_label",
        },
        {
            "Comparison": "In-processing: LinTS vs FairLinTS",
            "baseline_policy": "LinTS",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "FairLinTS_DP",
            "intervention_preprocessing": "uniform",
        },
    ],
    "adversarial_switching": [
        {
            "Comparison": "Preprocessing: EXP4 uniform vs reweighting",
            "baseline_policy": "EXP4",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "EXP4",
            "intervention_preprocessing": "reweigh_group_label",
        },
        {
            "Comparison": "In-processing: EXP4 vs FairEXP4",
            "baseline_policy": "EXP4",
            "baseline_preprocessing": "uniform",
            "intervention_policy": "FairEXP4_DP",
            "intervention_preprocessing": "uniform",
        },
    ],
}

imbalanced_significance_tables = export_synthetic_significance_tables(
    final_endpoint_df=imbalanced_final_endpoint_df,
    regimes=IMBALANCED_REGIMES,
    table_dir=IMBALANCED_TABLE_DIR,
    comparisons_by_regime=IMBALANCED_SIGNIFICANCE_COMPARISONS_BY_REGIME,
    significance_metrics=SIGNIFICANCE_METRICS,
    regime_labels=IMBALANCED_REGIME_LABELS,
    regime_file_tags=IMBALANCED_REGIME_FILE_TAGS,
    final_t=IMBALANCED_FINAL_T,
    policy_labels=POLICY_LABELS,
    preprocessing_labels=PREPROCESSING_LABELS,
    caption_prefix=(
        "Paired Wilcoxon tests with Holm correction for the synthetic "
        "regime-matched 80/20 sensitivity analysis"
    ),
    label_prefix="tab:synthetic",
    file_prefix="synthetic",
    seed_col="seed",
    alpha=0.05,
    value_digits=3,
)

## Generated artifacts

This final cell lists the generated figures, tables, and raw benchmark outputs.

It is mainly a reproducibility check: after running the notebook, the expected PNG, CSV, LaTeX, and trajectory files should be visible here.

In [ ]:
print("FIGURES")

for path in sorted(FIG_DIR.glob("synthetic_*.png")):
    print(path.name)

print("\nTABLES")

for path in sorted(TABLE_DIR.glob("synthetic_*")):
    print(path.name)

print("\nRAW RUN OUTPUTS")
print("Endpoint:", ENDPOINT_PATH, "exists =", ENDPOINT_PATH.exists())
print("Run index:", RUN_INDEX_PATH, "exists =", RUN_INDEX_PATH.exists())
print("Trajectories:", TRAJECTORY_DIR)

print("\n80/20 FIGURES")

for path in sorted(IMBALANCED_FIG_DIR.glob("synthetic_*.png")):
    print(path.name)

print("\n80/20 TABLES")

for path in sorted(IMBALANCED_TABLE_DIR.glob("synthetic_*")):
    print(path.name)

print("\n80/20 RAW RUN OUTPUTS")
print(
    "Endpoint:",
    IMBALANCED_ENDPOINT_PATH,
    "exists =",
    IMBALANCED_ENDPOINT_PATH.exists(),
)
print(
    "Run index:",
    IMBALANCED_RUN_INDEX_PATH,
    "exists =",
    IMBALANCED_RUN_INDEX_PATH.exists(),
)
print("Trajectories:", IMBALANCED_RUN_DIR / "trajectories")

In [ ]:
# ============================================================
# FIGURE 2 — Controlled synthetic contextual-bandit experiments
# Uses final_endpoint_df at T = 5000
# ============================================================




# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

REGIMES = [
    ("stationary_deterministic", "Stationary deterministic"),
    ("stationary_stochastic",    "Stationary stochastic"),
    ("adversarial_switching",    "Adversarial switching"),
]

POLICY_LABELS = {
    "LinUCB": "LinUCB",
    "FairLinUCB_DP": "FairLinUCB",
    "LinTS": "LinTS",
    "FairLinTS_DP": "FairLinTS",
    "EXP4": "EXP4",
    "FairEXP4_DP": "FairEXP4",
}

PREPROCESSING_LABELS = {
    "uniform": "U",
    "reweigh_group_label": "RW",
}

METRICS = ["average_reward", "DP_gap", "EO_gap"]


# ------------------------------------------------------------
# 2. Mean + Student-t 95% CI over the 50 seeds
# ------------------------------------------------------------

def mean_ci95(values):
    values = np.asarray(values, dtype=float)
    n = len(values)
    mean = values.mean()
    ci = t.ppf(0.975, n - 1) * values.std(ddof=1) / np.sqrt(n)
    return mean, ci


rows = []

for (regime, policy, preprocessing), g in final_endpoint_df.groupby(
    ["regime", "policy", "preprocessing"]
):
    row = {
        "regime": regime,
        "policy": policy,
        "preprocessing": preprocessing,
    }

    for metric in METRICS:
        mean, ci = mean_ci95(g[metric])
        row[f"{metric}_mean"] = mean
        row[f"{metric}_ci"] = ci

    rows.append(row)

summary = pd.DataFrame(rows)

display(summary.round(4))


# ------------------------------------------------------------
# 3. Figure 2 — 2 × 3
#
# Top:    Average reward vs DP gap
# Bottom: Average reward vs EO gap
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    3,
    figsize=(14.5, 8),
    sharey="row",
)

panel_letters = [
    ["A", "B", "C"],
    ["D", "E", "F"],
]


for col, (regime, regime_label) in enumerate(REGIMES):

    sub = summary[
        summary["regime"] == regime
    ]

    for row, fairness_metric in enumerate(["DP_gap", "EO_gap"]):

        ax = axes[row, col]

        for _, r in sub.iterrows():

            fair = r["policy"].startswith("Fair")
            reweighted = r["preprocessing"] == "reweigh_group_label"

            ax.errorbar(
                r["average_reward_mean"],
                r[f"{fairness_metric}_mean"],

                xerr=r["average_reward_ci"],
                yerr=r[f"{fairness_metric}_ci"],

                fmt="s" if reweighted else "o",
                markersize=8,
                markeredgewidth=1.2,
                zorder=3 if fair else 2,
                capsize=3,

                color="tab:red" if fair else "tab:blue",
            )

        ax.set_title(
            f"{panel_letters[row][col]}. {regime_label}",
            fontsize=11,
        )

        ax.set_xlabel(
            "Average reward\n(higher is better)"
        )

        if col == 0:
            ax.set_ylabel(
                "DP gap\n(lower is better)"
                if row == 0
                else
                "EO gap\n(lower is better)"
            )

        ax.grid(
            axis="both",
            alpha=0.20,
        )


# ------------------------------------------------------------
# 4. Common legend
# ------------------------------------------------------------

legend_elements = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="none",
        color="tab:blue",
        markersize=7,
        label="Standard-U",
    ),
    Line2D(
        [0], [0],
        marker="s",
        linestyle="none",
        color="tab:blue",
        markersize=7,
        label="Standard-RW",
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="none",
        color="tab:red",
        markersize=7,
        label="Fair-U",
    ),
    Line2D(
        [0], [0],
        marker="s",
        linestyle="none",
        color="tab:red",
        markersize=7,
        label="Fair-RW",
    ),
]

fig.legend(
    handles=legend_elements,
    loc="lower center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 0.02),
)


fig.suptitle(
    "Controlled synthetic contextual-bandit experiments",
    fontsize=14,
)


fig.text(
    0.5,
    0.075,
    (
        "U = uniform preprocessing; RW = group–label reweighting. "
        "Points represent means across 50 random seeds; "
        "horizontal and vertical error bars denote 95% confidence intervals."
    ),
    ha="center",
    fontsize=9,
)


fig.tight_layout(
    rect=[0, 0.11, 1, 0.95],
    h_pad=2.0,
    w_pad=1.5,
)


# ------------------------------------------------------------
# 5. Save
# ------------------------------------------------------------

PNG_PATH = FIG_DIR / "figure2_synthetic_tradeoffs_2x3.png"
SVG_PATH = FIG_DIR / "figure2_synthetic_tradeoffs_2x3.svg"

fig.savefig(
    PNG_PATH,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    SVG_PATH,
    bbox_inches="tight",
)

plt.show()

print("Saved:")
print(PNG_PATH)
print(SVG_PATH)

In [ ]:
# ============================================================
# FIGURE 3 — Temporal evolution of predictive utility
# and fairness for LinUCB / FairLinUCB
#
# Row 1: Adult
# Row 2: COMPAS
#
# Columns:
#   A/D = Average reward
#   B/E = DP gap
#   C/F = EO gap
#
# NO EXPERIMENT IS RERUN.
# ============================================================


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

ADULT_RUN_DIR = (
    RESULTS_ROOT
    / "adult_sex_cmab"
    / "full"
)

COMPAS_RUN_DIR = (
    RESULTS_ROOT
    / "compas_race_binary_fairness"
    / "full"
)

OUT_DIR = (
    RESULTS_ROOT
    / "comparative_figures"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 2. Find the LinUCB temporal file
# ------------------------------------------------------------

def find_linucb_temporal_file(run_dir):

    candidates = list(
        run_dir.rglob("temporal.csv")
    )

    print(f"\nSearching in: {run_dir}")

    valid = []

    for path in candidates:

        try:
            tmp = pd.read_csv(
                path,
                nrows=10,
            )

            cols = {
                str(c).lower()
                for c in tmp.columns
            }

            # A usable temporal file should contain
            # time, policy and fairness / reward information
            if (
                (
                    "t" in cols
                    or "round" in cols
                    or "horizon" in cols
                )
                and "policy" in cols
                and (
                    "average_reward" in cols
                    or "avg_reward" in cols
                )
            ):
                valid.append(path)

        except Exception:
            pass


    if len(valid) == 0:
        raise FileNotFoundError(
            f"No usable temporal.csv found under:\n{run_dir}"
        )


    # Prefer a path explicitly containing linucb
    linucb_paths = [
        p for p in valid
        if "linucb" in str(p).lower()
    ]


    if len(linucb_paths) > 0:
        selected = linucb_paths[0]
    else:
        selected = valid[0]


    print("Selected:", selected)

    return selected


ADULT_TEMPORAL_PATH = (
    find_linucb_temporal_file(
        ADULT_RUN_DIR
    )
)

COMPAS_TEMPORAL_PATH = (
    find_linucb_temporal_file(
        COMPAS_RUN_DIR
    )
)


# ------------------------------------------------------------
# 3. Load + normalize
# ------------------------------------------------------------

def prepare_linucb_temporal(
    path,
    dataset_name,
):

    df = pd.read_csv(
        path
    )

    # Normalize names if your package provides this helper
    if "normalize_metric_columns" in globals():
        df = normalize_metric_columns(
            df
        )


    # ------------------------------------------
    # Resolve time column
    # ------------------------------------------

    if "t" in df.columns:
        time_col = "t"

    elif "T" in df.columns:
        time_col = "T"

    elif "round" in df.columns:
        time_col = "round"

    elif "horizon" in df.columns:
        time_col = "horizon"

    else:
        raise KeyError(
            f"No temporal column found in {path}"
        )


    # ------------------------------------------
    # Resolve reward column
    # ------------------------------------------

    if "average_reward" in df.columns:
        reward_col = "average_reward"

    elif "avg_reward" in df.columns:
        reward_col = "avg_reward"

    else:
        raise KeyError(
            f"No reward column found in {path}"
        )


    # ------------------------------------------
    # Resolve fairness columns
    # ------------------------------------------

    if "DP_gap" in df.columns:
        dp_col = "DP_gap"

    elif "DP_gap_over_time" in df.columns:
        dp_col = "DP_gap_over_time"

    else:
        raise KeyError(
            f"No DP-gap column found in {path}"
        )


    if "EO_gap" in df.columns:
        eo_col = "EO_gap"

    elif "EO_gap_over_time" in df.columns:
        eo_col = "EO_gap_over_time"

    else:
        raise KeyError(
            f"No EO-gap column found in {path}"
        )


    # ------------------------------------------
    # Keep only LinUCB / FairLinUCB
    # ------------------------------------------

    wanted_policies = {
        "LinUCB",
        "FairLinUCB",
        "FairLinUCB_DP",
    }

    df = df[
        df["policy"].isin(
            wanted_policies
        )
    ].copy()


    # Harmonize fair policy name
    df["policy_display"] = (
        df["policy"]
        .replace(
            {
                "FairLinUCB_DP":
                    "FairLinUCB",
            }
        )
    )


    # ------------------------------------------
    # Keep only U / RW
    # ------------------------------------------

    wanted_preprocessing = {
        "uniform",
        "reweigh_group_label",
    }

    df = df[
        df["preprocessing"].isin(
            wanted_preprocessing
        )
    ].copy()


    out = pd.DataFrame(
        {
            "dataset":
                dataset_name,

            "seed":
                df["seed"],

            "t":
                pd.to_numeric(
                    df[time_col],
                    errors="coerce",
                ),

            "policy":
                df["policy_display"],

            "preprocessing":
                df["preprocessing"],

            "average_reward":
                pd.to_numeric(
                    df[reward_col],
                    errors="coerce",
                ),

            "DP_gap":
                pd.to_numeric(
                    df[dp_col],
                    errors="coerce",
                ),

            "EO_gap":
                pd.to_numeric(
                    df[eo_col],
                    errors="coerce",
                ),
        }
    )


    out = out.dropna(
        subset=[
            "t",
            "average_reward",
            "DP_gap",
            "EO_gap",
        ]
    )


    return out


adult_temporal = (
    prepare_linucb_temporal(
        ADULT_TEMPORAL_PATH,
        "Adult",
    )
)

compas_temporal = (
    prepare_linucb_temporal(
        COMPAS_TEMPORAL_PATH,
        "COMPAS",
    )
)


temporal = pd.concat(
    [
        adult_temporal,
        compas_temporal,
    ],
    ignore_index=True,
)


print("\nRows loaded:", len(temporal))

print("\nSeeds per configuration:")
display(
    temporal
    .groupby(
        [
            "dataset",
            "policy",
            "preprocessing",
        ]
    )["seed"]
    .nunique()
    .to_frame("n_seeds")
)


# ------------------------------------------------------------
# 4. Mean + 95% Student-t CI at each time point
# ------------------------------------------------------------

METRICS = [
    "average_reward",
    "DP_gap",
    "EO_gap",
]


rows = []


for (
    dataset,
    policy,
    preprocessing,
    horizon
), g in temporal.groupby(
    [
        "dataset",
        "policy",
        "preprocessing",
        "t",
    ]
):

    row = {
        "dataset":
            dataset,

        "policy":
            policy,

        "preprocessing":
            preprocessing,

        "t":
            horizon,

        "n":
            g["seed"].nunique(),
    }


    for metric in METRICS:

        values = (
            g[metric]
            .dropna()
            .to_numpy(
                dtype=float
            )
        )

        n = len(values)

        mean = values.mean()

        ci95 = (
            t.ppf(
                0.975,
                n - 1,
            )
            * values.std(
                ddof=1
            )
            / np.sqrt(n)
        )

        row[
            f"{metric}_mean"
        ] = mean

        row[
            f"{metric}_ci95"
        ] = ci95


    rows.append(row)


temporal_summary = pd.DataFrame(
    rows
)


# ------------------------------------------------------------
# 5. Optional plotting downsampling
#
# Keeps the figure lighter while preserving trajectories.
# Does NOT modify results.
# ------------------------------------------------------------

MAX_PLOT_POINTS = 150


def downsample_for_plot(df):

    unique_t = np.sort(
        df["t"].unique()
    )

    if len(unique_t) <= MAX_PLOT_POINTS:
        return df

    idx = np.linspace(
        0,
        len(unique_t) - 1,
        MAX_PLOT_POINTS,
    ).astype(int)

    selected_t = unique_t[
        np.unique(idx)
    ]

    return df[
        df["t"].isin(
            selected_t
        )
    ].copy()


temporal_plot = (
    temporal_summary
    .groupby(
        [
            "dataset",
            "policy",
            "preprocessing",
        ],
        group_keys=False,
    )
    .apply(
        downsample_for_plot
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 6. Visual encoding
# ------------------------------------------------------------

STYLE = {

    ("LinUCB", "uniform"):
        {
            "color": "tab:blue",
            "linestyle": "-",
            "label": "Standard-U",
        },

    ("LinUCB", "reweigh_group_label"):
        {
            "color": "tab:blue",
            "linestyle": ":",
            "label": "Standard-RW",
        },

    ("FairLinUCB", "uniform"):
        {
            "color": "tab:red",
            "linestyle": "-",
            "label": "Fair-U",
        },

    ("FairLinUCB", "reweigh_group_label"):
        {
            "color": "tab:red",
            "linestyle": ":",
            "label": "Fair-RW",
        },
}


# ------------------------------------------------------------
# 7. Figure 3 — 2 × 3
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15.5, 8.5),
    sharex=True,
)


PANEL_LETTERS = [
    ["A", "B", "C"],
    ["D", "E", "F"],
]


COLUMN_INFO = [

    (
        "average_reward",
        "Predictive utility",
        "Average reward",
    ),

    (
        "DP_gap",
        "Demographic parity",
        "DP gap",
    ),

    (
        "EO_gap",
        "Equalized odds",
        "EO gap",
    ),
]


DATASET_INFO = [
    ("Adult", 0),
    ("COMPAS", 1),
]


for dataset, row_index in DATASET_INFO:

    dataset_df = temporal_plot[
        temporal_plot["dataset"]
        == dataset
    ]


    for col_index, (
        metric,
        metric_title,
        ylabel,
    ) in enumerate(
        COLUMN_INFO
    ):

        ax = axes[
            row_index,
            col_index
        ]


        for (
            policy,
            preprocessing
        ), style in STYLE.items():

            sub = dataset_df[
                (
                    dataset_df["policy"]
                    == policy
                )
                &
                (
                    dataset_df[
                        "preprocessing"
                    ]
                    == preprocessing
                )
            ].sort_values(
                "t"
            )


            if len(sub) == 0:
                continue


            x = sub[
                "t"
            ].to_numpy()

            mean = sub[
                f"{metric}_mean"
            ].to_numpy()

            ci = sub[
                f"{metric}_ci95"
            ].to_numpy()


            ax.plot(
                x,
                mean,
                color=style["color"],
                linestyle=style["linestyle"],
                linewidth=1.8,
                label=style["label"],
            )


            ax.fill_between(
                x,
                mean - ci,
                mean + ci,
                color=style["color"],
                alpha=0.10,
                linewidth=0,
            )


        letter = (
            PANEL_LETTERS[
                row_index
            ][
                col_index
            ]
        )


        ax.set_title(
            f"{letter}. {dataset} — {metric_title}",
            fontsize=10.5,
        )


        ax.set_ylabel(
            ylabel
        )


        ax.grid(
            axis="y",
            alpha=0.20,
        )


        if row_index == 1:

            ax.set_xlabel(
                "Decision round t"
            )


# ------------------------------------------------------------
# 8. Common scales Adult / COMPAS for DP and EO
#
# Reward kept dataset-specific because Adult and COMPAS
# operate at substantially different levels.
# ------------------------------------------------------------

for col_index, metric in [
    (1, "DP_gap"),
    (2, "EO_gap"),
]:

    vals = temporal_summary[
        f"{metric}_mean"
    ]

    cis = temporal_summary[
        f"{metric}_ci95"
    ]

    ymin = max(
        0,
        (vals - cis).min()
    )

    ymax = (
        vals + cis
    ).max()

    padding = (
        0.05
        * (
            ymax - ymin
        )
    )

    for row_index in [0, 1]:

        axes[
            row_index,
            col_index
        ].set_ylim(
            max(
                0,
                ymin - padding
            ),
            ymax + padding,
        )


# ------------------------------------------------------------
# 9. One common legend
# ------------------------------------------------------------

handles, labels = (
    axes[0, 0]
    .get_legend_handles_labels()
)


fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(
        0.5,
        0.015,
    ),
)


# ------------------------------------------------------------
# 10. Global title and footer
# ------------------------------------------------------------

fig.suptitle(
    (
        "Temporal fairness and utility trajectories for LinUCB and FairLinUCB "
        
    ),
    fontsize=14,
    y=0.985,
)


fig.text(
    0.5,
    0.062,
    (
        "U = uniform preprocessing; "
        "RW = group–label reweighting. "
        "Lines represent means across 50 random seeds and "
        "shaded bands denote 95% confidence intervals."
    ),
    ha="center",
    fontsize=9,
)


fig.tight_layout(
    rect=[
        0,
        0.10,
        1,
        0.95,
    ],
    h_pad=2.2,
    w_pad=1.7,
)


# ------------------------------------------------------------
# 11. Save
# ------------------------------------------------------------

PNG_PATH = (
    OUT_DIR
    / "figure3_adult_compas_linucb_temporal_2x3.png"
)

SVG_PATH = (
    OUT_DIR
    / "figure3_adult_compas_linucb_temporal_2x3.svg"
)


fig.savefig(
    PNG_PATH,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    SVG_PATH,
    bbox_inches="tight",
)


plt.show()


print("\nSaved:")
print(PNG_PATH)
print(SVG_PATH)

In [ ]:
from pathlib import Path

# ============================================================
# Recover original synthetic benchmark paths
# ============================================================

OLD_SYNTHETIC_TABLE_DIR = Path(
    r"C:\Users\jmgre\Documents\Python_Tests\Fairness-JM-JN\notebooks"
    r"\results\full\synthetic_cmab_regime\full\tables"
)

ENDPOINT_PATH = (
    OLD_SYNTHETIC_TABLE_DIR
    / "endpoint_perseed.csv"
)

RUN_INDEX_PATH = (
    OLD_SYNTHETIC_TABLE_DIR
    / "run_index.csv"
)

print("ENDPOINT_PATH:")
print(ENDPOINT_PATH)
print("Exists:", ENDPOINT_PATH.exists())

print("\nRUN_INDEX_PATH:")
print(RUN_INDEX_PATH)
print("Exists:", RUN_INDEX_PATH.exists())

In [ ]:
# ============================================================
# Recover trajectory directory from ENDPOINT_PATH
# ============================================================

TRAJECTORY_SOURCE = (
    ENDPOINT_PATH
    .parent       # tables
    .parent       # full
    / "trajectories"
)

print("\nTRAJECTORY_SOURCE:")
print(TRAJECTORY_SOURCE)
print("Exists:", TRAJECTORY_SOURCE.exists())

In [ ]:
# ============================================================
# Minimal setup for supplementary synthetic figures
# ============================================================

# RESULTS_ROOT corresponds to:
# ...\results\full
RESULTS_ROOT_OLD = ENDPOINT_PATH.parents[3]

FIG_DIR = (
    RESULTS_ROOT_OLD
    / "final_figures"
    / "synthetic"
)

SUPP_DIR = (
    FIG_DIR
    / "supplementary"
)

SUPP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Synthetic full benchmark horizon
T_MAX = 5000

print("RESULTS_ROOT_OLD:")
print(RESULTS_ROOT_OLD)

print("\nFIG_DIR:")
print(FIG_DIR)

print("\nSUPP_DIR:")
print(SUPP_DIR)

print("\nT_MAX:", T_MAX)

In [ ]:
# ============================================================
# SUPPLEMENTARY FIGURES S1–S3
# Complete temporal trajectories — synthetic environments
#
# S1 = Stationary deterministic — LinUCB / FairLinUCB
# S2 = Stationary stochastic    — LinTS / FairLinTS
# S3 = Adversarial switching    — EXP4 / FairEXP4
#
# NO EXPERIMENT IS RERUN.
# ============================================================



# ============================================================
# 1. Recover ORIGINAL synthetic benchmark paths
# ============================================================

OLD_SYNTHETIC_TABLE_DIR = Path(
    r"C:\Users\jmgre\Documents\Python_Tests\Fairness-JM-JN\notebooks"
    r"\results\full\synthetic_cmab_regime\full\tables"
)

ENDPOINT_PATH = (
    OLD_SYNTHETIC_TABLE_DIR
    / "endpoint_perseed.csv"
)

RUN_INDEX_PATH = (
    OLD_SYNTHETIC_TABLE_DIR
    / "run_index.csv"
)

assert ENDPOINT_PATH.exists(), (
    f"endpoint_perseed.csv not found:\n{ENDPOINT_PATH}"
)

assert RUN_INDEX_PATH.exists(), (
    f"run_index.csv not found:\n{RUN_INDEX_PATH}"
)


# Saved trajectory directory
TRAJECTORY_SOURCE = (
    ENDPOINT_PATH
    .parent          # tables
    .parent          # full
    / "trajectories"
)

assert TRAJECTORY_SOURCE.exists(), (
    f"Trajectory directory not found:\n{TRAJECTORY_SOURCE}"
)


# ============================================================
# 2. Output directory
# ============================================================

# ...\results\full
RESULTS_ROOT_OLD = ENDPOINT_PATH.parents[3]

FIG_DIR = (
    RESULTS_ROOT_OLD
    / "final_figures"
    / "synthetic"
)

SUPP_DIR = (
    FIG_DIR
    / "supplementary"
)

SUPP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Full synthetic benchmark horizon
T_MAX = 5000

# Do not display the highly unstable earliest phase
ZOOM_START = 250


print("ENDPOINT_PATH:")
print(ENDPOINT_PATH)

print("\nTRAJECTORY_SOURCE:")
print(TRAJECTORY_SOURCE)

print("\nSUPP_DIR:")
print(SUPP_DIR)

print("\nT_MAX:", T_MAX)
print("ZOOM_START:", ZOOM_START)


# ============================================================
# 3. Synthetic configurations
# ============================================================

CONFIGS = {

    "S1": {
        "regime": "stationary_deterministic",
        "title": (
            "Temporal fairness and utility trajectories "
            "in the stationary deterministic synthetic environment"
        ),
        "standard": "LinUCB",
        "fair": "FairLinUCB_DP",
        "standard_label": "LinUCB",
        "fair_label": "FairLinUCB",
    },

    "S2": {
        "regime": "stationary_stochastic",
        "title": (
            "Temporal fairness and utility trajectories "
            "in the stationary stochastic synthetic environment"
        ),
        "standard": "LinTS",
        "fair": "FairLinTS_DP",
        "standard_label": "LinTS",
        "fair_label": "FairLinTS",
    },

    "S3": {
        "regime": "adversarial_switching",
        "title": (
            "Temporal fairness and utility trajectories "
            "in the adversarial switching synthetic environment"
        ),
        "standard": "EXP4",
        "fair": "FairEXP4_DP",
        "standard_label": "EXP4",
        "fair_label": "FairEXP4",
    },
}


PREPROCESSINGS = [
    "uniform",
    "reweigh_group_label",
]


# ============================================================
# 4. Read one saved trajectory
# ============================================================

def read_synthetic_trajectory(
    path,
    seed,
    policy,
    preprocessing,
):

    df = pd.read_csv(
        path,
        compression="gzip",
    )


    # --------------------------------------------------------
    # Decision-round coordinate
    # --------------------------------------------------------

    if "t" in df.columns:

        decision_round = pd.to_numeric(
            df["t"],
            errors="coerce",
        )

    elif "T" in df.columns:

        decision_round = pd.to_numeric(
            df["T"],
            errors="coerce",
        )

    elif "round" in df.columns:

        decision_round = pd.to_numeric(
            df["round"],
            errors="coerce",
        )

    else:

        # Saved temporal logs contain regularly spaced rows.
        n_rows = len(df)

        step = T_MAX / n_rows

        decision_round = (
            np.arange(
                1,
                n_rows + 1,
            )
            * step
        )


    # --------------------------------------------------------
    # Metric columns
    # --------------------------------------------------------

    if "avg_reward" in df.columns:
        reward_col = "avg_reward"

    elif "average_reward" in df.columns:
        reward_col = "average_reward"

    else:
        raise KeyError(
            f"No reward column found in:\n{path}"
        )


    if "DP_gap_over_time" in df.columns:
        dp_col = "DP_gap_over_time"

    elif "DP_gap" in df.columns:
        dp_col = "DP_gap"

    else:
        raise KeyError(
            f"No DP-gap column found in:\n{path}"
        )


    if "EO_gap_over_time" in df.columns:
        eo_col = "EO_gap_over_time"

    elif "EO_gap" in df.columns:
        eo_col = "EO_gap"

    else:
        raise KeyError(
            f"No EO-gap column found in:\n{path}"
        )


    out = pd.DataFrame(
        {
            "seed": seed,
            "policy": policy,
            "preprocessing": preprocessing,

            "t": np.asarray(
                decision_round,
                dtype=float,
            ),

            "average_reward": pd.to_numeric(
                df[reward_col],
                errors="coerce",
            ),

            "DP_gap": pd.to_numeric(
                df[dp_col],
                errors="coerce",
            ),

            "EO_gap": pd.to_numeric(
                df[eo_col],
                errors="coerce",
            ),
        }
    )


    return out.dropna(
        subset=[
            "t",
            "average_reward",
            "DP_gap",
            "EO_gap",
        ]
    )


# ============================================================
# 5. Load all 50 trajectories for one synthetic regime
# ============================================================

def load_regime_trajectories(
    config,
):

    regime = config["regime"]

    policies = [
        config["standard"],
        config["fair"],
    ]

    frames = []


    for preprocessing in PREPROCESSINGS:

        for policy in policies:

            policy_dir = (
                TRAJECTORY_SOURCE
                / regime
                / preprocessing
                / policy
            )

            print(
                "\nReading:",
                policy_dir
            )


            for seed in range(50):

                path = (
                    policy_dir
                    / f"seed_{seed:03d}.csv.gz"
                )


                if not path.exists():

                    raise FileNotFoundError(
                        f"Missing trajectory:\n{path}"
                    )


                frames.append(
                    read_synthetic_trajectory(
                        path=path,
                        seed=seed,
                        policy=policy,
                        preprocessing=preprocessing,
                    )
                )


            print("Files loaded: 50")


    return pd.concat(
        frames,
        ignore_index=True,
    )


# ============================================================
# 6. Pointwise mean + Student-t 95% CI
# ============================================================

METRICS = [
    "average_reward",
    "DP_gap",
    "EO_gap",
]


def summarize_temporal(
    df,
):

    rows = []


    for (
        policy,
        preprocessing,
        horizon,
    ), g in df.groupby(
        [
            "policy",
            "preprocessing",
            "t",
        ]
    ):

        row = {
            "policy": policy,
            "preprocessing": preprocessing,
            "t": horizon,
            "n": g["seed"].nunique(),
        }


        for metric in METRICS:

            values = (
                g[metric]
                .dropna()
                .to_numpy(
                    dtype=float
                )
            )

            n = len(values)

            mean = values.mean()


            if n > 1:

                ci95 = (
                    t.ppf(
                        0.975,
                        n - 1,
                    )
                    * values.std(
                        ddof=1
                    )
                    / np.sqrt(n)
                )

            else:

                ci95 = np.nan


            row[
                f"{metric}_mean"
            ] = mean

            row[
                f"{metric}_ci95"
            ] = ci95


        rows.append(row)


    return pd.DataFrame(
        rows
    )


# ============================================================
# 7. Plot one supplementary figure
# ============================================================

def make_synthetic_supp_figure(
    figure_number,
    config,
):

    print(
        "\n"
        + "=" * 70
    )

    print(
        f"Generating {figure_number}: "
        f"{config['regime']}"
    )

    print(
        "=" * 70
    )


    # --------------------------------------------------------
    # Load raw trajectories
    # --------------------------------------------------------

    raw = load_regime_trajectories(
        config
    )


    # --------------------------------------------------------
    # Summarize across 50 seeds
    # --------------------------------------------------------

    summary = summarize_temporal(
        raw
    )


    # --------------------------------------------------------
    # Remove highly unstable initial phase from VISUALIZATION
    # --------------------------------------------------------

    summary_plot = (
        summary[
            summary["t"] >= ZOOM_START
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Sanity check
    # --------------------------------------------------------

    print(
        f"\n{figure_number} seeds per configuration:"
    )

    display(
        raw
        .groupby(
            [
                "policy",
                "preprocessing",
            ]
        )["seed"]
        .nunique()
        .to_frame("n_seeds")
    )


    # --------------------------------------------------------
    # Styles
    # --------------------------------------------------------

    styles = {

        (
            config["standard"],
            "uniform",
        ): {
            "color": "tab:blue",
            "linestyle": "-",
            "label": "Standard-U",
        },

        (
            config["standard"],
            "reweigh_group_label",
        ): {
            "color": "tab:blue",
            "linestyle": ":",
            "label": "Standard-RW",
        },

        (
            config["fair"],
            "uniform",
        ): {
            "color": "tab:red",
            "linestyle": "-",
            "label": "Fair-U",
        },

        (
            config["fair"],
            "reweigh_group_label",
        ): {
            "color": "tab:red",
            "linestyle": ":",
            "label": "Fair-RW",
        },
    }


    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(14.5, 4.5),
    )


    panels = [

        (
            "average_reward",
            "Predictive utility",
            "Average reward",
        ),

        (
            "DP_gap",
            "Demographic parity",
            "DP gap",
        ),

        (
            "EO_gap",
            "Equalized odds",
            "EO gap",
        ),
    ]


    for idx, (
        metric,
        title,
        ylabel,
    ) in enumerate(
        panels
    ):

        ax = axes[idx]


        for (
            policy,
            preprocessing,
        ), style in styles.items():

            g = (
                summary_plot[
                    (
                        summary_plot["policy"]
                        == policy
                    )
                    &
                    (
                        summary_plot["preprocessing"]
                        == preprocessing
                    )
                ]
                .sort_values("t")
            )


            x = (
                g["t"]
                .to_numpy()
            )

            y = (
                g[
                    f"{metric}_mean"
                ]
                .to_numpy()
            )

            ci = (
                g[
                    f"{metric}_ci95"
                ]
                .to_numpy()
            )


            ax.plot(
                x,
                y,
                color=style["color"],
                linestyle=style["linestyle"],
                linewidth=1.8,
                label=style["label"],
            )


            ax.fill_between(
                x,
                y - ci,
                y + ci,
                color=style["color"],
                alpha=0.10,
                linewidth=0,
            )


        # ----------------------------------------------------
        # Panel formatting
        # ----------------------------------------------------

        letter = chr(
            ord("A") + idx
        )


        ax.set_title(
            f"{letter}. {title}",
            fontsize=10.5,
        )

        ax.set_xlabel(
            r"Decision round $t$"
        )

        ax.set_ylabel(
            ylabel
        )

        ax.grid(
            axis="y",
            alpha=0.20,
        )

        ax.set_xlim(
            ZOOM_START,
            T_MAX,
        )


        # ----------------------------------------------------
        # Fairness metrics cannot be negative.
        # Also avoid early-CI-driven giant scales.
        # ----------------------------------------------------

        if metric in [
            "DP_gap",
            "EO_gap",
        ]:

            upper = (
                summary_plot[
                    f"{metric}_mean"
                ]
                +
                summary_plot[
                    f"{metric}_ci95"
                ]
            ).max()

            upper = max(
                float(upper) * 1.08,
                0.02,
            )

            ax.set_ylim(
                0,
                upper,
            )


    # --------------------------------------------------------
    # Fixed legend order
    # --------------------------------------------------------

    legend_elements = [

        Line2D(
            [0], [0],
            color="tab:blue",
            linestyle="-",
            linewidth=1.8,
            label="Standard-U",
        ),

        Line2D(
            [0], [0],
            color="tab:blue",
            linestyle=":",
            linewidth=1.8,
            label="Standard-RW",
        ),

        Line2D(
            [0], [0],
            color="tab:red",
            linestyle="-",
            linewidth=1.8,
            label="Fair-U",
        ),

        Line2D(
            [0], [0],
            color="tab:red",
            linestyle=":",
            linewidth=1.8,
            label="Fair-RW",
        ),
    ]


    fig.legend(
        handles=legend_elements,
        loc="lower center",
        ncol=4,
        frameon=False,
        bbox_to_anchor=(
            0.5,
            0.025,
        ),
        fontsize=9.5,
    )


    # --------------------------------------------------------
    # Main title
    # --------------------------------------------------------

    fig.suptitle(
        config["title"],
        fontsize=13.5,
        y=0.98,
    )


    # --------------------------------------------------------
    # Footer
    # --------------------------------------------------------

    fig.text(
        0.5,
        0.085,
        (
            "U = uniform preprocessing; "
            "RW = group–label reweighting. "
            "Lines represent means across 50 random seeds and "
            "shaded bands denote 95% confidence intervals."
        ),
        ha="center",
        fontsize=8.8,
    )


    fig.tight_layout(
        rect=[
            0,
            0.15,
            1,
            0.93,
        ],
        w_pad=1.8,
    )


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    base_name = (
        f"figure_{figure_number}_"
        f"{config['regime']}_temporal"
    )


    png_path = (
        SUPP_DIR
        / f"{base_name}.png"
    )

    svg_path = (
        SUPP_DIR
        / f"{base_name}.svg"
    )


    fig.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
    )

    fig.savefig(
        svg_path,
        bbox_inches="tight",
    )


    plt.show()


    print(
        "\nSaved:",
        png_path
    )

    print(
        "Saved:",
        svg_path
    )


# ============================================================
# 8. Generate S1, S2, S3
# ============================================================

for figure_number in [
    "S1",
    "S2",
    "S3",
]:

    make_synthetic_supp_figure(
        figure_number,
        CONFIGS[figure_number],
    )

In [ ]:
# ============================================================
# SUPPLEMENTARY FIGURE S10
# Synthetic sensitivity analysis — 80/20 group imbalance
#
# Row 1: Stationary deterministic — LinUCB / FairLinUCB
# Row 2: Stationary stochastic    — LinTS / FairLinTS
# Row 3: Adversarial switching    — EXP4 / FairEXP4
#
# Columns:
#   A/D/G = Average reward
#   B/E/H = DP gap
#   C/F/I = EO gap
#
# NO EXPERIMENT IS RERUN.
# ============================================================




# ============================================================
# 1. Locate original 80/20 saved trajectories
# ============================================================

# RESULTS_ROOT_OLD was defined in the S1–S3 code as:
# ...\notebooks\results\full

IMBALANCED_RUN_DIR = (
    RESULTS_ROOT_OLD
    / "synthetic_cmab_sensitivity_80_20_regime"
    / "full"
)

IMBALANCED_ENDPOINT_PATH = (
    IMBALANCED_RUN_DIR
    / "tables"
    / "endpoint_perseed.csv"
)

IMBALANCED_RUN_INDEX_PATH = (
    IMBALANCED_RUN_DIR
    / "tables"
    / "run_index.csv"
)

IMBALANCED_TRAJECTORY_SOURCE = (
    IMBALANCED_RUN_DIR
    / "trajectories"
)


print("80/20 endpoint:")
print(IMBALANCED_ENDPOINT_PATH)
print("Exists:", IMBALANCED_ENDPOINT_PATH.exists())

print("\n80/20 run index:")
print(IMBALANCED_RUN_INDEX_PATH)
print("Exists:", IMBALANCED_RUN_INDEX_PATH.exists())

print("\n80/20 trajectories:")
print(IMBALANCED_TRAJECTORY_SOURCE)
print("Exists:", IMBALANCED_TRAJECTORY_SOURCE.exists())


assert IMBALANCED_ENDPOINT_PATH.exists(), (
    f"80/20 endpoint file not found:\n"
    f"{IMBALANCED_ENDPOINT_PATH}"
)

assert IMBALANCED_TRAJECTORY_SOURCE.exists(), (
    f"80/20 trajectory directory not found:\n"
    f"{IMBALANCED_TRAJECTORY_SOURCE}"
)


# ============================================================
# 2. Regime configuration
# ============================================================

IMBALANCED_CONFIGS = [

    {
        "regime": "stationary_deterministic",
        "regime_label": "Stationary deterministic",
        "standard": "LinUCB",
        "fair": "FairLinUCB_DP",
    },

    {
        "regime": "stationary_stochastic",
        "regime_label": "Stationary stochastic",
        "standard": "LinTS",
        "fair": "FairLinTS_DP",
    },

    {
        "regime": "adversarial_switching",
        "regime_label": "Adversarial switching",
        "standard": "EXP4",
        "fair": "FairEXP4_DP",
    },
]


# ============================================================
# 3. Load one complete 80/20 regime
# ============================================================

def load_imbalanced_regime(
    config,
):

    regime = config["regime"]

    policies = [
        config["standard"],
        config["fair"],
    ]

    frames = []


    for preprocessing in [
        "uniform",
        "reweigh_group_label",
    ]:

        for policy in policies:

            policy_dir = (
                IMBALANCED_TRAJECTORY_SOURCE
                / regime
                / preprocessing
                / policy
            )

            print(
                "\nReading:",
                policy_dir
            )


            for seed in range(50):

                path = (
                    policy_dir
                    / f"seed_{seed:03d}.csv.gz"
                )

                if not path.exists():

                    raise FileNotFoundError(
                        f"Missing 80/20 trajectory:\n{path}"
                    )


                frames.append(
                    read_synthetic_trajectory(
                        path=path,
                        seed=seed,
                        policy=policy,
                        preprocessing=preprocessing,
                    )
                )


            print("Files loaded: 50")


    raw = pd.concat(
        frames,
        ignore_index=True,
    )


    print(
        f"\n{regime} — seeds per configuration:"
    )

    display(
        raw
        .groupby(
            [
                "policy",
                "preprocessing",
            ]
        )["seed"]
        .nunique()
        .to_frame("n_seeds")
    )


    return raw


# ============================================================
# 4. Load and summarize all three regimes
# ============================================================

imbalanced_summaries = {}


for config in IMBALANCED_CONFIGS:

    raw = load_imbalanced_regime(
        config
    )

    summary = summarize_temporal(
        raw
    )

    # Same visualization window as S1–S3
    summary = (
        summary[
            summary["t"] >= ZOOM_START
        ]
        .copy()
    )

    imbalanced_summaries[
        config["regime"]
    ] = summary


# ============================================================
# 5. Visual style
# ============================================================

def curve_style(
    policy,
    preprocessing,
):

    fair = policy.startswith("Fair")

    reweighted = (
        preprocessing
        == "reweigh_group_label"
    )

    return {
        "color":
            "tab:red"
            if fair
            else "tab:blue",

        "linestyle":
            ":"
            if reweighted
            else "-",
    }


legend_elements = [

    Line2D(
        [0], [0],
        color="tab:blue",
        linestyle="-",
        linewidth=1.8,
        label="Standard-U",
    ),

    Line2D(
        [0], [0],
        color="tab:blue",
        linestyle=":",
        linewidth=1.8,
        label="Standard-RW",
    ),

    Line2D(
        [0], [0],
        color="tab:red",
        linestyle="-",
        linewidth=1.8,
        label="Fair-U",
    ),

    Line2D(
        [0], [0],
        color="tab:red",
        linestyle=":",
        linewidth=1.8,
        label="Fair-RW",
    ),
]


# ============================================================
# 6. Create 3 × 3 figure
# ============================================================

fig, axes = plt.subplots(
    3,
    3,
    figsize=(15.5, 11.5),
)


METRIC_INFO = [

    (
        "average_reward",
        "Predictive utility",
        "Average reward",
    ),

    (
        "DP_gap",
        "Demographic parity",
        "DP gap",
    ),

    (
        "EO_gap",
        "Equalized odds",
        "EO gap",
    ),
]


PANEL_LETTERS = [
    ["A", "B", "C"],
    ["D", "E", "F"],
    ["G", "H", "I"],
]


for row_index, config in enumerate(
    IMBALANCED_CONFIGS
):

    regime = config["regime"]

    summary = (
        imbalanced_summaries[
            regime
        ]
    )


    for col_index, (
        metric,
        metric_title,
        ylabel,
    ) in enumerate(
        METRIC_INFO
    ):

        ax = axes[
            row_index,
            col_index
        ]


        for policy in [
            config["standard"],
            config["fair"],
        ]:

            for preprocessing in [
                "uniform",
                "reweigh_group_label",
            ]:

                g = (
                    summary[
                        (
                            summary["policy"]
                            == policy
                        )
                        &
                        (
                            summary[
                                "preprocessing"
                            ]
                            == preprocessing
                        )
                    ]
                    .sort_values("t")
                )


                style = curve_style(
                    policy,
                    preprocessing,
                )


                x = (
                    g["t"]
                    .to_numpy()
                )

                y = (
                    g[
                        f"{metric}_mean"
                    ]
                    .to_numpy()
                )

                ci = (
                    g[
                        f"{metric}_ci95"
                    ]
                    .to_numpy()
                )


                ax.plot(
                    x,
                    y,
                    color=style["color"],
                    linestyle=style["linestyle"],
                    linewidth=1.8,
                )


                ax.fill_between(
                    x,
                    y - ci,
                    y + ci,
                    color=style["color"],
                    alpha=0.10,
                    linewidth=0,
                )


        # ----------------------------------------------------
        # Panel title
        # ----------------------------------------------------

        letter = (
            PANEL_LETTERS[
                row_index
            ][
                col_index
            ]
        )


        ax.set_title(
            (
                f"{letter}. "
                f"{config['regime_label']} — "
                f"{metric_title}"
            ),
            fontsize=10.3,
        )


        # ----------------------------------------------------
        # Axes
        # ----------------------------------------------------

        ax.set_xlim(
            ZOOM_START,
            T_MAX,
        )

        ax.set_xlabel(
            r"Decision round $t$"
        )

        ax.set_ylabel(
            ylabel
        )

        ax.grid(
            axis="y",
            alpha=0.20,
        )


        # Fairness gaps cannot be negative
        if metric in [
            "DP_gap",
            "EO_gap",
        ]:

            upper = (
                summary[
                    f"{metric}_mean"
                ]
                +
                summary[
                    f"{metric}_ci95"
                ]
            ).max()

            upper = max(
                float(upper) * 1.08,
                0.02,
            )

            ax.set_ylim(
                0,
                upper,
            )


# ============================================================
# 7. Global title, legend and footer
# ============================================================

fig.suptitle(
    (
        "Temporal fairness and utility trajectories "
        "under 80/20 protected-group imbalance"
    ),
    fontsize=14,
    y=0.985,
)


fig.legend(
    handles=legend_elements,
    loc="lower center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(
        0.5,
        0.018,
    ),
    fontsize=9.5,
)


fig.text(
    0.5,
    0.055,
    (
        "The majority and minority protected groups represent "
        "80% and 20% of observations, respectively. "
        "U = uniform preprocessing; RW = group–oracle-action "
        "reweighting. Lines represent means across 50 random seeds "
        "and shaded bands denote 95% confidence intervals."
    ),
    ha="center",
    fontsize=8.8,
)


fig.tight_layout(
    rect=[
        0,
        0.09,
        1,
        0.955,
    ],
    h_pad=2.2,
    w_pad=1.6,
)


# ============================================================
# 8. Save as Supplementary Figure S10
# ============================================================

S10_PNG = (
    SUPP_DIR
    / "figure_S10_synthetic_80_20_temporal_3x3.png"
)

S10_SVG = (
    SUPP_DIR
    / "figure_S10_synthetic_80_20_temporal_3x3.svg"
)


fig.savefig(
    S10_PNG,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    S10_SVG,
    bbox_inches="tight",
)


plt.show()


print("\nSaved:")
print(S10_PNG)
print(S10_SVG)